
# CustomerInsight AI — Customer Feature Engineering

**Research Phase 03: RFM Analysis**

## Research objective

Our original dataset contains individual product
transactions. We want to convert these records
into customer-level features.

We will use Recency, Frequency and Monetary (RFM)
to describe different aspects of purchasing behaviour.

### Research questions

1. How recently do customers make purchases?
2. How often do identified customers purchase?
3. How much do they spend on positive sales?
4. Are the feature distributions suitable for clustering?

The results are descriptive. They do not establish
which clustering algorithm performs best.


In [5]:

from pathlib import Path

import pandas as pd
import plotly.express as px

from customerinsight.retail import (
    load_retail_data,
    data_quality_summary,
)
from customerinsight.features import (
    build_rfm,
    describe_rfm,
    prepare_customer_sales,
)

# Find the project root whether VS Code starts here or in notebooks.
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/raw/Online Retail.xlsx"

transactions = load_retail_data(DATA_PATH)

print(f"Original transaction rows: {len(transactions):,}")
display(pd.DataFrame(
    data_quality_summary(transactions).items(),
    columns=["Quality check", "Value"]
))


Original transaction rows: 541,909


,Quality check,Value
0,rows,541909
1,columns,10
2,unique_customers,4372
3,missing_customer_ids,135080
4,duplicate_rows,5268
5,cancellation_rows,9288
6,negative_quantities,10624
7,nonpositive_prices,2517
8,invalid_dates,0



## Preparing customer transactions

Not every transaction row is suitable for initial RFM
analysis.

For this baseline, we use positive sales with a
valid customer ID and invoice ID.

We preserve the original data so cancellations,
returns and excluded records can be investigated later.

**Why this matters:** Removing records changes the
population represented by the analysis.


In [6]:

customer_sales = prepare_customer_sales(transactions)

excluded_rows = len(transactions) - len(customer_sales)

print(f"Rows used for customer features: {len(customer_sales):,}")
print(f"Rows excluded from this baseline: {excluded_rows:,}")
print(f"Identified customers: {customer_sales['customer_id'].nunique():,}")

display(customer_sales.head())


Rows used for customer features: 397,884
Rows excluded from this baseline: 144,025
Identified customers: 4,338


,invoice_id,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_cancellation,line_total
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34



## Creating Recency, Frequency and Monetary features

- **Recency:** Days since the customer's latest purchase.
- **Frequency:** Distinct positive-sale invoices.
- **Monetary:** Total value of positive purchase lines.

We use one day after the latest observed valid sale
as the reference date.

This ensures we analyze historical purchasing behaviour
without depending on the current calendar date.



## Visualization 1 — How recently did customers purchase?

### Why are we using a histogram?

A histogram helps us understand how common different
recency values are across the customer population.

**How to read it:** The horizontal axis represents
days since the last purchase. The vertical axis
represents the number of customers.

A smaller recency means the customer purchased
more recently.

**Research significance:** This feature may help
clustering models distinguish recently active
customers from less recently active ones.


In [7]:

fig = px.histogram(
    rfm,
    x="recency",
    nbins=40,
    title="How recently did customers make a purchase?",
    labels={
        "recency": "Days since last purchase",
        "count": "Number of customers",
    },
)

fig.update_layout(
    yaxis_title="Number of customers",
    bargap=0.05,
)

fig.show()

print(
    "Median days since last purchase:",
    rfm["recency"].median()
)


NameError: name 'rfm' is not defined


## Visualization 2 — How frequently do customers purchase?

### Why are we using a histogram?

We want to understand whether most customers purchase
only once or return multiple times.

**How to read it:** Each bar represents a range of
distinct invoice counts.

**Research significance:** Differences in frequency
may help us identify customers with recurring
purchase behaviour.

**Limitation:** Frequency depends on the time period
covered by the dataset.


In [ ]:

# Limit the display range without removing data from our RFM table.
frequency_limit = rfm["frequency"].quantile(0.99)

frequency_view = rfm.loc[
    rfm["frequency"] <= frequency_limit
]

fig = px.histogram(
    frequency_view,
    x="frequency",
    nbins=35,
    title="How often do customers make purchases?",
    labels={
        "frequency": "Number of distinct purchase invoices",
    },
)

fig.update_layout(
    yaxis_title="Number of customers",
)

fig.show()

print("Median purchase frequency:", rfm["frequency"].median())
print(
    "Customers shown in the chart:",
    f"{len(frequency_view):,} of {len(rfm):,}"
)



## Visualization 3 — How much do customers spend?

### Why use a histogram?

Customer spending may be uneven. Some customers
purchase low-value items, while others make
much larger purchases.

A histogram reveals the distribution instead
of showing only the average.

**How to read it:** The horizontal axis represents
customer spending in GBP (£).

**Research significance:** A highly skewed spending
distribution may influence distance-based clustering.

**Limitation:** We show customers up to the 99th
percentile for readability. Extreme observations
remain in our original RFM dataset.


In [ ]:

spending_limit = rfm["monetary"].quantile(0.99)

spending_view = rfm.loc[
    rfm["monetary"] <= spending_limit
]

fig = px.histogram(
    spending_view,
    x="monetary",
    nbins=50,
    title="How much do customers spend on positive sales?",
    labels={
        "monetary": "Gross positive customer spending (£)",
    },
)

fig.update_layout(
    yaxis_title="Number of customers",
)

fig.show()

print(f"Median positive spending: £{rfm['monetary'].median():,.2f}")
print(f"Mean positive spending: £{rfm['monetary'].mean():,.2f}")
print(
    "Customers shown in the chart:",
    f"{len(spending_view):,} of {len(rfm):,}"
)


In [ ]:

OUTPUT_PATH = PROJECT_ROOT / "data/processed/customer_rfm.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save the features so later experiments use the same input data.
rfm.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(rfm):,} customers to:")
print(OUTPUT_PATH)



## Research observations

### Recency
Record the median and distribution observed.

### Frequency
Describe how often customers tend to purchase.

### Monetary
Record whether customer spending appears skewed.

### Feature engineering decisions
Explain why we selected positive sales, distinct
invoices and a fixed observation reference date.

### Limitations
Our baseline excludes customers without identifiers
and does not reconcile refunds against original sales.

RFM also cannot fully capture product preferences,
seasonality or changes in customer behaviour.

### Next research step
Investigate feature scaling and clustering methods,
including whether preprocessing choices affect
segmentation quality and stability.
